In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.preprocessing import StandardScaler

# Set a random seed so your results never change randomly (Required by rubric)
np.random.seed(42)

# Define column names (The dataset has 26 columns, no headers)
cols = ['engine_id', 'cycle', 'op1', 'op2', 'op3'] + [f'sensor_{i}' for i in range(1, 22)]

# Load the FD001 Training Data - Use raw string with 'r' prefix
train_data = pd.read_csv(r"C:\Users\kouro\Jet_Engine_Project\data\train_FD001.txt", sep='\s+', header=None, names=cols)

print("Data loaded successfully!")
print(f"Total rows: {len(train_data)}")
print(f"Total engines: {train_data['engine_id'].nunique()}")

Data loaded successfully!
Total rows: 20631
Total engines: 100


In [2]:
def prepare_training_data(df):
    # Find out when each engine died (max cycle)
    max_cycles = df.groupby('engine_id')['cycle'].max()
    df['max_cycle'] = df['engine_id'].map(max_cycles)
    
    # Calculate Remaining Useful Life (RUL)
    df['RUL'] = df['max_cycle'] - df['cycle']
    
    # Create classification labels: 1 if failing in 30 cycles, 0 if not
    df['fail_in_30'] = (df['RUL'] <= 30).astype(int)
    
    # Drop the temporary column
    df = df.drop('max_cycle', axis=1)
    return df

train_data = prepare_training_data(train_data)
print("Answers (RUL and Classification Labels) created!")

Answers (RUL and Classification Labels) created!


In [3]:
# Get a list of all 100 engines
all_engines = train_data['engine_id'].unique()
np.random.shuffle(all_engines)

# Split them: 70 for Training, 30 for Validation (tuning)
train_ids = all_engines[:70]
val_ids = all_engines[70:]

# Actually divide the data table based on these IDs
train_df = train_data[train_data['engine_id'].isin(train_ids)].copy()
val_df = train_data[train_data['engine_id'].isin(val_ids)].copy()

print(f"Training on {len(train_ids)} engines. Validating on {len(val_ids)} engines.")

Training on 70 engines. Validating on 30 engines.


In [4]:
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]

# 1. Scale the data (Learn only on Train, apply to Val)
scaler = StandardScaler()
train_df[sensor_cols] = scaler.fit_transform(train_df[sensor_cols])
val_df[sensor_cols] = scaler.transform(val_df[sensor_cols])

# Save the scaler to a file! We need this for the web app later.
joblib.dump(scaler, 'scaler.pkl')
print("Scaler saved as scaler.pkl")

# 2. Create Window Features (Trailing 5-cycle average)
def add_features(df):
    df_grouped = df.groupby('engine_id')[sensor_cols]
    # Calculate average of the last 5 cycles
    rolling_mean = df_grouped.rolling(window=5, min_periods=1).mean().reset_index(0, drop=True)
    rolling_mean.columns = [f"{col}_mean" for col in sensor_cols]
    return pd.concat([df, rolling_mean], axis=1)

train_df = add_features(train_df)
val_df = add_features(val_df)

print("Features engineered successfully!")

Scaler saved as scaler.pkl
Features engineered successfully!


In [5]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

# Define the columns the AI is allowed to look at
features = sensor_cols + [f"{col}_mean" for col in sensor_cols]

print("Training Random Forest Regression Model...")
rf_model = RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42)
rf_model.fit(train_df[features], train_df['RUL'])

# Test it on the Validation set
val_preds = rf_model.predict(val_df[features])
mae = mean_absolute_error(val_df['RUL'], val_preds)
print(f"Validation MAE: {mae:.2f} cycles off on average")

# Save the model!
joblib.dump(rf_model, 'rul_model.pkl')
print("Model saved as rul_model.pkl")

Training Random Forest Regression Model...
Validation MAE: 32.78 cycles off on average
Model saved as rul_model.pkl


In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score

print("Training Logistic Regression (30-cycle warning)...")
clf_model = LogisticRegression(max_iter=1000, random_state=42)
clf_model.fit(train_df[features], train_df['fail_in_30'])

# Test it
val_clf_preds = clf_model.predict(val_df[features])
precision = precision_score(val_df['fail_in_30'], val_clf_preds)
print(f"Validation Precision: {precision:.2f}")

# Save the model!
joblib.dump(clf_model, 'clf_model.pkl')
print("Model saved as clf_model.pkl")

Training Logistic Regression (30-cycle warning)...
Validation Precision: 0.90
Model saved as clf_model.pkl


In [7]:
# 1. Load the official test data using your exact path!
test_data = pd.read_csv(r"C:\Users\kouro\Jet_Engine_Project\data\test_FD001.txt", sep='\s+', header=None, names=cols)

# 2. Load the official answer key (RUL_FD001.txt)
true_rul = pd.read_csv(r"C:\Users\kouro\Jet_Engine_Project\data\RUL_FD001.txt", sep='\s+', header=None, names=['RUL'])
true_rul['engine_id'] = true_rul.index + 1  # Add engine IDs 1 to 100

# 3. Calculate the actual exact RUL for every row in the test set
# Find the last cycle recorded for each engine in the test set
max_cycles_test = test_data.groupby('engine_id')['cycle'].max().reset_index()
max_cycles_test.columns = ['engine_id', 'last_cycle']

# Merge the answer key with the last observed cycle
rul_math = pd.merge(max_cycles_test, true_rul, on='engine_id')
rul_math['death_cycle'] = rul_math['last_cycle'] + rul_math['RUL']

# Map the death cycle back to the main test dataframe
test_data = pd.merge(test_data, rul_math[['engine_id', 'death_cycle']], on='engine_id')

# Current RUL = Death Cycle - Current Cycle
test_data['RUL'] = test_data['death_cycle'] - test_data['cycle']
test_data['fail_in_30'] = (test_data['RUL'] <= 30).astype(int)

# Drop the temporary column
test_data = test_data.drop('death_cycle', axis=1)

# 4. Apply our Scaler and Feature Engineering (The exact same ones from Training)
test_data[sensor_cols] = scaler.transform(test_data[sensor_cols])
test_data = add_features(test_data)

print("Official Test Data loaded, labeled, and processed!")

Official Test Data loaded, labeled, and processed!


In [8]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, precision_score
import numpy as np

# Test the Regression Model (RUL)
final_rul_preds = rf_model.predict(test_data[features])
final_mae = mean_absolute_error(test_data['RUL'], final_rul_preds)

# Calculate RMSE manually (since squared parameter might not be supported)
final_rmse = np.sqrt(mean_squared_error(test_data['RUL'], final_rul_preds))

# Alternative: Calculate RMSE directly
final_rmse = np.sqrt(((test_data['RUL'] - final_rul_preds) ** 2).mean())

# Test the Classification Model (30-day warning)
final_clf_preds = clf_model.predict(test_data[features])
final_precision = precision_score(test_data['fail_in_30'], final_clf_preds)

print("=== FINAL EXAM RESULTS (Put these in your report!) ===")
print(f"Regression MAE: {final_mae:.2f}")
print(f"Regression RMSE: {final_rmse:.2f}")
print(f"Classification Precision (30-cycle): {final_precision:.2f}")

=== FINAL EXAM RESULTS (Put these in your report!) ===
Regression MAE: 35.78
Regression RMSE: 47.06
Classification Precision (30-cycle): 0.81
